In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

import sys; sys.path.insert(0, "../src")
from quad_fuzzy import make_data, fit_predictors, embed_2d, pick_corners, barycentric_triangle, quad_weights, delaunay_weights


for noise in (0.03, 0.5):
    print(f"noise={noise}:", {k: round(v, 3) for k, v in fit_predictors(make_data(noise=noise)).items()})

noise=0.03: {'a': 0.914, 'b': 0.903, 'c': 0.915, 'd': 0.967}
noise=0.5: {'a': 0.081, 'b': 0.031, 'c': 0.044, 'd': 0.131}


In [2]:
from sklearn.decomposition import PCA

df = make_data(noise=0.03)
X = df[["a", "b", "c", "d"]]

for k in (2, 3):
    pca = PCA(n_components=k).fit(X)
    kept = pca.explained_variance_ratio_.sum()
    print(f"{k}D projection keeps {kept:.1%} of variance")

# the actual 2D coordinates we'll draw triangles on later:
pca2 = PCA(n_components=2)
xy = pca2.fit_transform(X)      # shape (500, 2) — each 4D point squashed to a plane
print("\nfirst 5 projected points:\n", xy[:5])

2D projection keeps 70.8% of variance
3D projection keeps 99.8% of variance

first 5 projected points:
 [[ 0.56381585 -0.41479207]
 [-0.31821328  0.48357387]
 [ 0.05896536  0.6094032 ]
 [ 0.45265945  0.31530543]
 [-0.01800437 -0.42264001]]


In [3]:
## Data resembles  1 - (a+b+c)/3 + tiny noise ,, it looks like a 3 dimensional object located in a 4 dimensional space.
#fidelity depends on whether the flattened position still reflects the point's true location — and that's decided by how much information the flattening threw away.
# using 2d model for human explainability

In [4]:
import numpy as np
from scipy.spatial import Delaunay

def coverage_of_minimal_simplex(dim, n_points=2000, seed=0):
    """
    Fraction of a scattered cloud that lands INSIDE a single minimal simplex.
    A minimal simplex in `dim` dimensions has dim+1 corners.
    We pick those corners from the cloud itself (farthest-point spread) so the
    simplex is a fair 'best minimal container', not a random one.
    """
    rng = np.random.default_rng(seed)
    pts = rng.uniform(0, 1, size=(n_points, dim))   # scattered cloud in the unit cube

    # farthest-point sampling to pick dim+1 well-spread corners
    k = dim + 1
    chosen = [int(rng.integers(n_points))]
    dist = np.linalg.norm(pts - pts[chosen[0]], axis=1)
    for _ in range(k - 1):
        nxt = int(np.argmax(dist))
        chosen.append(nxt)
        dist = np.minimum(dist, np.linalg.norm(pts - pts[nxt], axis=1))

    corners = pts[chosen]
    tri = Delaunay(corners)                    # a single simplex (dim+1 corners)
    inside = tri.find_simplex(pts) >= 0        # True where a point is inside it
    return inside.mean()

print(f"{'dim':>4} | {'corners':>7} | {'coverage':>9}")
print("-" * 26)
for dim in range(2, 9):
    cov = coverage_of_minimal_simplex(dim)
    print(f"{dim:>4} | {dim+1:>7} | {cov:>8.1%}")

 dim | corners |  coverage
--------------------------
   2 |       3 |    42.9%
   3 |       4 |    12.8%
   4 |       5 |     4.2%
   5 |       6 |     0.4%
   6 |       7 |     0.5%
   7 |       8 |     0.4%
   8 |       9 |     0.4%


In [5]:
from quad_fuzzy import make_data, embed_2d
df = make_data()
pca, xy = embed_2d(df)
idx = pick_corners(xy, 4)      # your just-written function, defined in the cell above
print("corner indices:", idx)
print("corner coords:\n", xy[idx])
print("cloud x-range:", xy[:,0].min().round(2), xy[:,0].max().round(2))
print("cloud y-range:", xy[:,1].min().round(2), xy[:,1].max().round(2))

corner indices: [425, 108, 220, 234]
corner coords:
 [[-0.26148287  0.00574472]
 [ 0.82445614  0.09927323]
 [ 0.28496289 -0.61330746]
 [-0.39338591  0.72293062]]
cloud x-range: -0.8 0.83
cloud y-range: -0.65 0.72


In [6]:
A, B, C = np.array([0,0]), np.array([1,0]), np.array([0,1])
print(barycentric_triangle([0,0],     A, B, C))   # on corner A -> expect (1,0,0)
print(barycentric_triangle([1/3,1/3], A, B, C))   # centroid    -> expect (~.33,~.33,~.33)
print(barycentric_triangle([1,1],     A, B, C))   # OUTSIDE      -> expect a NEGATIVE weight

(1.0, 0.0, 0.0)
(0.3333333333333334, 0.3333333333333333, 0.3333333333333333)
(-1.0, 1.0, 1.0)


In [7]:
from quad_fuzzy import make_data, embed_2d, pick_corners
df = make_data(); pca, xy = embed_2d(df)
idx = pick_corners(xy, 4)
corners = xy[idx]
centroid = corners.mean(axis=0)          # dead center of the 4 corners -> inside
faraway  = np.array([100.0, 100.0])      # miles away -> outside both halves

print("inside :", quad_weights(centroid, corners))
print("outside:", quad_weights(faraway,  corners))

inside : None
outside: None


In [8]:
A, B, C, D = corners
print("centroid:", centroid)
print("raw ABC weights:", barycentric_triangle(centroid, A, B, C))
print("sum:", sum(barycentric_triangle(centroid, A, B, C)))

centroid: [0.11363756 0.05366028]
raw ABC weights: (0.666206223877598, 0.35722456049832085, -0.023430784375918844)
sum: 0.9999999999999999


In [9]:
print("corners shape:", np.asarray(corners).shape)   # expect (4, 2)
print("type of one corner:", type(corners[0]))
print("barycentric on a corner:", barycentric_triangle(corners[0], A, B, C))  # p=A -> expect (1,0,0)

corners shape: (4, 2)
type of one corner: <class 'numpy.ndarray'>
barycentric on a corner: (1.0, 0.0, -0.0)


In [10]:
A, B, C, D = corners
# a point that is 50% A + 30% B + 20% C -> guaranteed inside triangle ABC
inside_pt = 0.5*A + 0.3*B + 0.2*C
print("inside :", quad_weights(inside_pt, corners))   # expect ~{A:.5, B:.3, C:.2, D:0}
print("outside:", quad_weights(np.array([100.0,100.0]), corners))  # expect None

inside : {'A': 0.5, 'B': 0.3, 'C': 0.19999999999999996, 'D': 0.0}
outside: None


In [11]:
from scipy.spatial import Delaunay
df = make_data(); pca, xy = embed_2d(df)
idx = pick_corners(xy, 10)          # 10 corners
corners = xy[idx]
tri = Delaunay(corners)

w = delaunay_weights(xy[0], tri, len(corners))
print("weights:", None if w is None else np.round(w, 3))
print("how many nonzero:", None if w is None else np.count_nonzero(w))

weights: None
how many nonzero: None


In [12]:
# find the points that ARE covered, and inspect one
covered = [i for i in range(len(xy)) if tri.find_simplex(xy[i]) >= 0]
print(f"covered: {len(covered)} / {len(xy)}  ({len(covered)/len(xy):.1%})")

i = covered[0]                      # a point known to be inside
w = delaunay_weights(xy[i], tri, len(corners))
print("weights:", np.round(w, 3))
print("nonzero :", np.count_nonzero(w))

covered: 467 / 500  (93.4%)
weights: [0.273 0.    0.    0.586 0.    0.    0.    0.    0.141 0.   ]
nonzero : 3


In [13]:
from sklearn.linear_model import LinearRegression

# the black box: predicts d from a,b,c (the model we're explaining)
blackbox = LinearRegression().fit(df[["a","b","c"]], df["d"])
bb_pred  = blackbox.predict(df[["a","b","c"]])   # black box's d for every point

corner_d = df["d"].values[idx]      # each corner's d-value = its rule output

fid_errors = []
for i in covered:                    # only points that HAVE an explanation
    w = delaunay_weights(xy[i], tri, len(corners))
    rule_blend = np.dot(w, corner_d)          # explanation's predicted d
    fid_errors.append(abs(rule_blend - bb_pred[i]))   # gap vs the BLACK BOX

print(f"mean fidelity error: {np.mean(fid_errors):.4f}")

mean fidelity error: 0.0338


In [14]:
def measure(noise, k=10, seed=0):
    df = make_data(noise=noise, seed=seed)
    pca, xy = embed_2d(df)
    idx = pick_corners(xy, k)
    corners = xy[idx]
    tri = Delaunay(corners)

    blackbox = LinearRegression().fit(df[["a","b","c"]], df["d"])
    bb_pred  = blackbox.predict(df[["a","b","c"]])
    corner_d = df["d"].values[idx]

    covered = [i for i in range(len(xy)) if tri.find_simplex(xy[i]) >= 0]
    coverage = len(covered) / len(xy)

    fid = []
    for i in covered:
        w = delaunay_weights(xy[i], tri, len(corners))
        fid.append(abs(np.dot(w, corner_d) - bb_pred[i]))
    fidelity = np.mean(fid)

    mean_r2 = np.mean(list(fit_predictors(df).values()))
    return mean_r2, coverage, fidelity

print(f"{'noise':>6} | {'meanR²':>7} | {'coverage':>9} | {'fidelity':>9}")
print("-"*40)
for noise in (0.03, 0.10, 0.20, 0.35, 0.50):
    r2, cov, fid = measure(noise)
    print(f"{noise:>6} | {r2:>7.3f} | {cov:>8.1%} | {fid:>9.4f}")

 noise |  meanR² |  coverage |  fidelity
----------------------------------------
  0.03 |   0.925 |    93.4% |    0.0338
   0.1 |   0.553 |    92.2% |    0.0729
   0.2 |   0.263 |    91.8% |    0.1377
  0.35 |   0.121 |    93.4% |    0.2697
   0.5 |   0.072 |    93.4% |    0.3851
